In [ ]:
import os
from langchain_openai import ChatOpenAI
#获取api_key
api_key = os.getenv('ARK_API_KEY')
print(api_key)

In [ ]:
model = ChatOpenAI(
    openai_api_base="https://ark.cn-beijing.volces.com/api/v3",
    openai_api_key=api_key,	# app_key
    model_name="doubao-seed-1-6-flash-250828",	# 您想使用的特定模型的名称或标识符。
    max_tokens=1000, #限制响应中的令牌总数，有效控制输出长度。
    temperature= 0.7,  #控制模型输出的随机性。值越高，响应越具创造性；值越低，响应越确定性。
    timeout=30, #模型的响应时间s
)
response = model.invoke("介绍一下你自己")

In [ ]:
response.content

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    {"role": "system", "content": "你是一个将中文翻译成英文的有用助手。"},
    {"role": "user", "content":"翻译：我喜欢编程。"},
    {"role": "assistant","content":"I like programming"},
    {"role": "user", "content":"翻译：我喜欢构建应用程序。"}
]


conversation1 = [
    SystemMessage("你是一个将英语翻译成法语的有用助手。"),
    HumanMessage("翻译：我喜欢编程。"),
    AIMessage("J'adore la programmation。"),
    HumanMessage("翻译：我喜欢构建应用程序。")
]


response = model.invoke(conversation1)
response.content

In [ ]:
for chunk in model.stream("天空是什么颜色？"):
    print(chunk.text, end="|", flush=True)

In [ ]:
for chunk in model.stream("天空是什么颜色？"):
    for block in chunk.content_blocks:
        if block["type"] == "reasoning" and (reasoning := block.get("reasoning")):
            print(f"推理：{reasoning}")
        elif block["type"] == "tool_call_chunk":
            print(f"工具调用块：{block}")
        elif block["type"] == "text":
            print(block["text"])
        

In [ ]:
full = None  # None | AIMessageChunk
for chunk in model.stream("天空是什么颜色？"):
    full = chunk if full is None else full + chunk
    print(full.text)

print(full.content_blocks)
# [{"type": "text", "text": "天空通常是蓝色..."}]

In [ ]:
async for event in model.astream_events("你好"):

    if event["event"] == "on_chat_model_start":
        print(f"输入：{event['data']['input']}")

    elif event["event"] == "on_chat_model_stream":
        print(f"令牌：{event['data']['chunk'].text}")

    elif event["event"] == "on_chat_model_end":
        print(f"完整消息：{event['data']['output'].text}")

    else:
        pass

In [ ]:
responses = model.batch([
    "为什么鹦鹉有五颜六色的羽毛？",
    "飞机是如何飞行的？",
    "什么是量子计算？"
],
    config={
        'max_concurrency':5,
                       })
n = 1
for response in responses:
    print(n)
    ++n
    print(response)

In [ ]:
model.batch(
    list_of_inputs,
    config={
        'max_concurrency': 5,  # 限制为 5 个并行调用
    }
)

In [ ]:
#工具调用
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """获取某个位置的天气。"""
    return f"{location} 天气晴朗。"

model_with_tools = model.bind_tools([get_weather])  # [!code highlight]

response = model_with_tools.invoke("波士顿的天气怎么样？")
for tool_call in response.tool_calls:
    # 查看模型发出的工具调用
    print(f"工具：{tool_call['name']}")
    print(f"参数：{tool_call['args']}")
response

In [ ]:
get_weather.invoke( response.tool_calls[0]["args"]["location"])

In [ ]:
model.invoke("波士顿的天气怎么样？")

In [ ]:
model_with_tools = model.bind_tools([get_weather])

message = [{"role":"user","content":"波士顿的天气怎样？"}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

print(message)
final_response = model_with_tools.invoke(message)

In [ ]:
final_response.content

In [ ]:
response = model_with_tools.invoke(
     "波士顿和东京的天气怎么样？"
)
print(response.tool_calls)

#执行所有工具
result=[]
for tool_call in response.tool_calls:
    if tool_call['name'] == 'get__weather':
        result = get_weather.invoke(tool_call)
    result.append(result)
        

In [6]:
# ollma model connnection

from langchain_ollama import ChatOllama
ollama_model = ChatOllama(model="qwen2.5:3b")
response = ollama_model.invoke("介绍一下你自己")
print(response)

content='当然，很高兴为您介绍自己。\n\n我是阿里云研发的大型语言模型，能够回答各种问题、提供信息和帮助解决问题。我由阿里巴巴集团旗下的阿里达科技股份有限公司开发，致力于为用户提供一个强大而友好的人工智能助手。\n\n我的训练数据涵盖了广泛的文本类型，包括但不限于书籍、报纸文章、期刊文章、网页内容等。通过这些丰富的数据来源，我可以生成连贯的文本，理解和回应用户的问题，同时还能帮助进行创作任务如续写故事或诗歌。\n\n我还能够完成一些特定的任务，比如解谜游戏、计算问题、编写代码片段、甚至是回答开放性问题和撰写文档。当然，在使用我的过程中，请确保您的行为遵守当地的法律法规以及道德准则。\n\n总的来说，我是阿里云研发的一个强大且多功能的语言模型AI助手，致力于为用户提供高效、准确的帮助和支持。' additional_kwargs={} response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-02-13T05:21:04.6539494Z', 'done': True, 'done_reason': 'stop', 'total_duration': 21332484500, 'load_duration': 4866334900, 'prompt_eval_count': 31, 'prompt_eval_duration': 999261000, 'eval_count': 172, 'eval_duration': 15019934100, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'} id='lc_run--019c5571-bd37-7c90-8c09-44c8157a9458-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 31, 'output_tokens': 172, 'total_tokens': 203}
